# How Python's data structures work, and what they cost

A gentle tour of `list`, `deque`, `dict`, `set`, sorting and `bisect`: **what** each structure is, **why** some operations are fast and others slow, and **how** the inside of the structure explains the cost.

Heaps have their own notebook: [`heaps.ipynb`](heaps.ipynb).

> Timings in this notebook come from whatever machine ran it last. Your numbers will differ; look at how they **grow** as `n` doubles, not at the absolute values.

## Prologue: the one idea behind everything

Memory is a long street of numbered houses. Jumping to house number 4,000,017 is instant: you know the address, you go there. That is **O(1)**.

What's slow is anything that makes you **walk the street**: checking house after house (**O(n)**), or moving residents from house to house. Do that for every item, n times over, and you get **O(n²)**.

Almost every speed question in Python comes down to two questions:

1. **Do I know the address, or do I have to search?**
2. **Does this operation make things move?**

## 1. `list`: a row of houses with numbered doors

A list is a **dynamic array**: one contiguous block of memory holding **pointers** to your objects. Every slot is 8 bytes, so slot `i` lives at `start + i × 8`. That's why `lst[i]` is O(1): it's arithmetic, not a search.

```
list  ──▶ [ ptr | ptr | ptr | ptr | ptr |  ·  |  ·  |  ·  ]
             ▼     ▼     ▼     ▼     ▼    ← spare capacity
            'a'    42   obj   3.14  None
```

### 1.1 Why `append` is fast: spare capacity

The list keeps **extra empty slots** at the end, so appending just fills the next one. When they run out, Python allocates a bigger block and copies every pointer across. That copy is O(n), but it's rare, so the average cost per append is constant: **amortized O(1)**.

You can watch the over-allocation happen. `sys.getsizeof` only changes when the list grabs a bigger block:

In [1]:
import sys

lst = []
empty = sys.getsizeof(lst)
last = empty
print(f"len=0    bytes={empty}")
for i in range(1, 65):
    lst.append(i)
    size = sys.getsizeof(lst)
    if size != last:   # a reallocation just happened
        print(f"len={i:<4} bytes={size:<4} -> room for {(size - empty) // 8} pointers")
        last = size

len=0    bytes=56
len=1    bytes=88   -> room for 4 pointers
len=5    bytes=120  -> room for 8 pointers
len=9    bytes=184  -> room for 16 pointers
len=17   bytes=248  -> room for 24 pointers
len=25   bytes=312  -> room for 32 pointers
len=33   bytes=376  -> room for 40 pointers
len=41   bytes=472  -> room for 52 pointers
len=53   bytes=568  -> room for 64 pointers


The size jumps only every so often, and each jump buys room for several more appends. Between jumps, `append` just writes a pointer.

### 1.2 Why `pop(0)` is expensive: everything shifts

Slot 0 must always hold the first element, because that's what makes `lst[i]` simple arithmetic. So removing slot 0 means **every other pointer moves one house to the left**:

```
before: [ A | B | C | D | E ]
pop(0):   ✕   ←   ←   ←   ←      every element moves
after:  [ B | C | D | E | · ]
```

That's O(n) per pop. `pop()` from the **end** moves nobody, so it's O(1).

In [2]:
lst = ["A", "B", "C", "D", "E"]
print("pop()  ->", lst.pop(), lst)    # from the end: nothing moves
print("pop(0) ->", lst.pop(0), lst)   # from the front: B, C, D shift left

pop()  -> E ['A', 'B', 'C', 'D']
pop(0) -> A ['B', 'C', 'D']


Using a list as a queue (append at the back, `pop(0)` at the front) costs O(n) per step, so O(n²) to drain it. Compare with a `deque` (§2): watch what happens as `n` doubles.

In [3]:
import time
from collections import deque

def drain_list(n):
    lst = list(range(n))
    t = time.perf_counter()
    while lst:
        lst.pop(0)
    return time.perf_counter() - t

def drain_deque(n):
    dq = deque(range(n))
    t = time.perf_counter()
    while dq:
        dq.popleft()
    return time.perf_counter() - t

for n in (10_000, 20_000, 40_000):
    print(f"n={n:>6}:  list.pop(0) {drain_list(n) * 1000:8.1f} ms    "
          f"deque.popleft() {drain_deque(n) * 1000:6.2f} ms")

n= 10000:  list.pop(0)      5.6 ms    deque.popleft()   0.22 ms
n= 20000:  list.pop(0)     32.4 ms    deque.popleft()   0.63 ms
n= 40000:  list.pop(0)    120.2 ms    deque.popleft()   1.09 ms


Each time `n` doubles, the list time grows about **4×** (that's O(n²)), and the deque time about **2×** (O(n)).

### 1.3 Slices copy

`lst[a:b]` builds a **new list**, so it costs O(b − a). Changing the slice doesn't change the original. Slicing inside recursion (`helper(lst[1:])`) copies at every level, so pass indexes instead.

In [4]:
a = [1, 2, 3, 4]
b = a[1:]          # a new list: copies 3 pointers
b[0] = 99
print("a =", a)
print("b =", b)

a = [1, 2, 3, 4]
b = [99, 3, 4]


### 1.4 List costs at a glance

| Operation | Cost | Why |
|---|---|---|
| `lst[i]`, `lst[i] = x` | O(1) | Address arithmetic |
| `append`, `pop()` | O(1) amortized | Spare capacity at the end |
| `pop(0)`, `insert(0, x)` | O(n) | Everything shifts |
| `x in lst`, `lst.index(x)` | O(n) | No address for a value, so walk and compare |
| `lst[a:b]` | O(b − a) | Copies into a new list |
| `lst.sort()` | O(n log n) | Timsort (§4) |
| `len(lst)` | O(1) | Stored, not counted |

## 2. `deque`: a chain of small boxes, open at both ends

`collections.deque` (say "deck") is a **doubly linked list of fixed-size blocks**. In CPython each block holds 64 slots, and each block knows its left and right neighbours. The deque remembers where the leftmost and rightmost items are.

```
        ┌─────────┐    ┌─────────┐    ┌─────────┐
 ◀──────│ · · A B │◀──▶│ C D E F │◀──▶│ G H · · │──────▶
        └─────────┘    └─────────┘    └─────────┘
           ▲ left index                    ▲ right index
```

**Why `popleft` is O(1):** read the item at the left index, then move the index one step right. Nobody else moves. An emptied block is unhooked from the chain. The list had to keep slot 0 at a fixed address; the deque's "front" is just a moving bookmark.

**What you give up:** indexing into the middle. `dq[5000]` has to walk block by block from the nearer end, so it's O(n).

| Operation | `list` | `deque` |
|---|---|---|
| append / pop at the right | O(1) | O(1) |
| append / pop at the left | **O(n)** | **O(1)** |
| index in the middle | **O(1)** | O(n) |

In [5]:
from collections import deque

dq = deque(["B", "C"])
dq.appendleft("A")      # O(1)
dq.append("D")          # O(1)
print(dq)
print("popleft ->", dq.popleft(), dq)
print("pop     ->", dq.pop(), dq)

deque(['A', 'B', 'C', 'D'])
popleft -> A deque(['B', 'C', 'D'])
pop     -> D deque(['B', 'C'])


### 2.1 The classic use: BFS

Every breadth-first search wants a first-in, first-out queue, so every BFS should use a deque:

In [6]:
from collections import deque

graph = {
    "A": ["B", "C"],
    "B": ["D"],
    "C": ["D", "E"],
    "D": ["F"],
    "E": ["F"],
    "F": [],
}

def bfs_order(graph, start):
    q = deque([start])
    seen = {start}            # set membership: O(1)
    order = []
    while q:
        node = q.popleft()    # O(1), not lst.pop(0)
        order.append(node)
        for nxt in graph[node]:
            if nxt not in seen:
                seen.add(nxt)
                q.append(nxt)
    return order

print(bfs_order(graph, "A"))

['A', 'B', 'C', 'D', 'E', 'F']


### 2.2 A free sliding window: `maxlen`

With `maxlen=k`, appending to a full deque silently drops the item at the other end. That's "the last k events" for free:

In [7]:
from collections import deque

last3 = deque(maxlen=3)
for event in ["login", "view", "click", "buy", "logout"]:
    last3.append(event)
    print(f"{event:<7} -> {list(last3)}")

login   -> ['login']
view    -> ['login', 'view']
click   -> ['login', 'view', 'click']
buy     -> ['view', 'click', 'buy']
logout  -> ['click', 'buy', 'logout']


## 3. `dict` and `set`: a magic address book

A list finds things by **position**. A dict finds things by **key**, and makes that a direct jump rather than a walk.

### 3.1 How: hashing turns a key into an address

1. `hash(key)` turns the key into a big integer. The same key always gives the same integer.
2. That integer, reduced to the table size, picks a **slot** in an internal array.
3. The key and value are stored at that slot.

To read `d[key]`, Python repeats steps 1–2 and goes straight there: **O(1) on average**.

Small integers hash to themselves, which makes the idea easy to see. With a table of 8 slots, `hash(k) % 8` picks the slot:

In [8]:
table_size = 8
for k in (3, 4, 11, 19):
    print(f"key={k:<3} hash={hash(k):<3} slot={hash(k) % table_size}")

key=3   hash=3   slot=3


key=4   hash=4   slot=4
key=11  hash=11  slot=3
key=19  hash=19  slot=3


Keys 3, 11 and 19 all want slot 3: that's a **collision**. (String hashes are randomized each time Python starts, for security, so they aren't shown here.)

### 3.2 Collisions and resizing

CPython uses **open addressing**: if the slot is taken, it follows a deterministic sequence of other slots (a pseudo-random probe derived from the hash) until it finds a free one. A lookup follows the same sequence until it finds the matching key.

To keep those probe chains short, the dict **never gets too full**: at about two-thirds full it grows and re-inserts everything. That rehash is O(n) but rare, so inserts stay **amortized O(1)**, just like `list.append`. You can see the resizes:

In [9]:
import sys

d = {}
last = sys.getsizeof(d)
for i in range(1, 200):
    d[i] = i
    size = sys.getsizeof(d)
    if size != last:
        print(f"inserting key #{i:<4}: {last:>5} -> {size:>5} bytes (table grew)")
        last = size

inserting key #1   :    64 ->   224 bytes (table grew)
inserting key #6   :   224 ->   352 bytes (table grew)
inserting key #11  :   352 ->   632 bytes (table grew)
inserting key #22  :   632 ->  1168 bytes (table grew)
inserting key #43  :  1168 ->  2264 bytes (table grew)
inserting key #86  :  2264 ->  4688 bytes (table grew)
inserting key #171 :  4688 ->  9304 bytes (table grew)


### 3.3 Why keys must be hashable (immutable)

If a key could change after you stored it, its hash would change, and Python would look in the wrong slot. That's why strings, numbers and **tuples** can be keys, but lists and dicts can't. To use a list or a grid state as a key, convert it to a tuple.

In [10]:
d = {}
try:
    d[[1, 2]] = "visited"
except TypeError as e:
    print("TypeError:", e)

d[(1, 2)] = "visited"       # a tuple works
print(d)

TypeError: unhashable type: 'list'
{(1, 2): 'visited'}


### 3.4 The modern dict remembers insertion order

Since Python 3.7, a dict is two parts: a compact array of entries **in insertion order**, plus a small hash table of indexes into it. Iteration walks the entry array, so you get insertion order for free. Updating a value keeps the key's position.

In [11]:
d = {}
d["banana"] = 3
d["apple"] = 7
d["cherry"] = 1
d["banana"] = 30          # update: keeps its original position
print(list(d.items()))

[('banana', 30), ('apple', 7), ('cherry', 1)]


### 3.5 `set`: a dict with only keys

Same hash-table idea, no values, so `x in s` is **O(1)**, while `x in lst` is O(n). Turning a list into a set costs O(n) once; after that, every membership test is nearly free.

In [12]:
import time

n = 100_000
as_list = list(range(n))
as_set = set(as_list)
missing = -1           # worst case for the list: it checks every element
lookups = 200

t = time.perf_counter()
for _ in range(lookups):
    missing in as_list
list_us = (time.perf_counter() - t) / lookups * 1e6

t = time.perf_counter()
for _ in range(lookups):
    missing in as_set
set_us = (time.perf_counter() - t) / lookups * 1e6

print(f"x in list: {list_us:10.2f} µs per lookup")
print(f"x in set : {set_us:10.2f} µs per lookup")

x in list:     510.70 µs per lookup
x in set :       0.30 µs per lookup


| Operation | dict / set | Why |
|---|---|---|
| `d[k]`, `k in d`, `d[k] = v` | O(1) average | Hash, then jump |
| `del d[k]` | O(1) average | Hash, then mark the slot |
| Iterate | O(n) | Walk the entries |
| Key with the max value | O(n) | No ordering by value |

(In theory, if every key collided, lookups would degrade to O(n). Good hash functions make that vanishingly rare with normal data.)

### 3.6 Helpful relatives: `Counter` and `defaultdict`

In [13]:
from collections import Counter, defaultdict

words = "the cat and the hat and the bat".split()
counts = Counter(words)                 # O(n)
print(counts)
print("top 2:", counts.most_common(2))  # uses a heap internally

edges = [("a", "b"), ("a", "c"), ("b", "c")]
adj = defaultdict(list)                 # a missing key starts as []
for u, v in edges:
    adj[u].append(v)
print(dict(adj))

Counter({'the': 3, 'and': 2, 'cat': 1, 'hat': 1, 'bat': 1})
top 2: [('the', 3), ('and', 2)]
{'a': ['b', 'c'], 'b': ['c']}


In [ ]:
# Conters
from collections import Coutner
counts=Counter({'p2': 3, 'p1': 3, 'p3': 2, 'p4': 1})
# >>> counts['p1']
# 3
# >>> counts.most_common(2)
# [('p2', 3), ('p1', 3)]

counts=Counter({'p2': (3,-200), 'p1': (3,-100), 'p3': (2,-150), 'p4': (1,-30)})
counts.most_common(1)

## 4. Sorting, including "how do I sort a dict?"

### 4.1 Timsort

`sorted()` and `list.sort()` use **Timsort**, a hybrid of merge sort and insertion sort. It looks for **runs** (stretches already in order, or reversed) and merges them.

- Worst and average case: **O(n log n)**.
- Already or nearly sorted: close to **O(n)**, because the input is one long run.
- **Stable**: items that compare equal keep their original order.

`lst.sort()` sorts in place and returns `None`; `sorted(x)` returns a new list and accepts any iterable.

In [14]:
import random
import time

n = 1_000_000
shuffled = random.sample(range(n), n)
already_sorted = sorted(shuffled)

for name, data in [("shuffled", shuffled), ("already sorted", already_sorted)]:
    t = time.perf_counter()
    sorted(data)
    print(f"{name:<15} {(time.perf_counter() - t) * 1000:7.1f} ms")

shuffled          284.8 ms
already sorted     12.3 ms


### 4.2 Stability lets you sort in passes

To sort by department, and by score (highest first) within each department: sort by the **secondary** key first, then by the **primary** key. The second sort keeps the first sort's order among ties.

In [15]:
people = [("ana", "eng", 82), ("bo", "ops", 95), ("cy", "eng", 91), ("di", "ops", 70)]

by_score = sorted(people, key=lambda p: p[2], reverse=True)   # secondary key
by_dept = sorted(by_score, key=lambda p: p[1])                 # primary key
for p in by_dept:
    print(p)

('cy', 'eng', 91)
('ana', 'eng', 82)
('bo', 'ops', 95)
('di', 'ops', 70)


### 4.3 Sorting a dict

A dict isn't sorted; it's ordered by insertion. "Sorting a dict" really means **sorting its items**, and optionally building a new dict in that order.

The `key=` function runs **once per item** (Python computes all the keys first), so it's cheap. Cost: O(n log n) for the sort, plus O(n) to rebuild the dict.

In [16]:
scores = {"ana": 82, "bo": 95, "cy": 82, "di": 70}

print(sorted(scores))                                            # keys, alphabetically
print(sorted(scores.items(), key=lambda kv: kv[1]))              # by value, ascending
print(sorted(scores.items(), key=lambda kv: kv[1], reverse=True))  # by value, descending
print(sorted(scores.items(), key=lambda kv: (-kv[1], kv[0])))    # value desc, then name asc

ranked = dict(sorted(scores.items(), key=lambda kv: kv[1], reverse=True))
print(ranked)

['ana', 'bo', 'cy', 'di']
[('di', 70), ('ana', 82), ('cy', 82), ('bo', 95)]
[('bo', 95), ('ana', 82), ('cy', 82), ('di', 70)]
[('bo', 95), ('ana', 82), ('cy', 82), ('di', 70)]
{'bo': 95, 'ana': 82, 'cy': 82, 'di': 70}


If you only need the top few, don't sort everything. A heap gives O(n log k) instead of O(n log n) (see [`heaps.ipynb`](heaps.ipynb)):

In [17]:
import heapq

scores = {"ana": 82, "bo": 95, "cy": 82, "di": 70}
print(heapq.nlargest(2, scores.items(), key=lambda kv: kv[1]))

[('bo', 95), ('ana', 82)]


### 4.4 `bisect`: binary search in a sorted list

`bisect` halves the search range repeatedly, so it finds a position in **O(log n)**. But `insort` still has to **insert**, which shifts elements (§1.2), so keeping a big list sorted under many inserts is O(n) per insert. Use `bisect` for **lookups** in sorted data; use a heap for "keep giving me the smallest".

In [18]:
import bisect

breakpoints = [60, 70, 80, 90]
grades = "FDCBA"
marks = [33, 99, 77, 70, 89, 90, 100]
print([grades[bisect.bisect(breakpoints, m)] for m in marks])

sorted_vals = [1, 3, 3, 5, 8, 13]
x = 4
print(f"values <= {x}:", bisect.bisect_right(sorted_vals, x))
print(f"insert position for {x}:", bisect.bisect_left(sorted_vals, x))

bisect.insort(sorted_vals, 4)    # O(log n) to find + O(n) to shift
print(sorted_vals)

['F', 'A', 'C', 'C', 'B', 'A', 'A']
values <= 4: 3
insert position for 4: 3
[1, 3, 3, 4, 5, 8, 13]


## 5. Small things that bite

- **Strings are immutable.** `s += "x"` in a loop *can* build a new string every time (O(n²) overall). CPython sometimes optimizes this in place, but don't rely on it: collect the pieces in a list and `"".join(...)` once.
- **`x in` on a list inside a loop** is O(n²). Use a set.
- **Slices copy** (§1.3).
- **Recursion depth** defaults to about 1,000 frames. A deep DFS on a big input may need an explicit stack (a list with `append`/`pop`).
- **Tuples** are hashable and lighter than lists: the natural key for coordinates and states, e.g. `(row, col)`.

In [19]:
import sys

parts = []
for i in range(5):
    parts.append(str(i))
print("".join(parts))                    # one O(n) join

print("recursion limit:", sys.getrecursionlimit())

# iterative DFS with an explicit stack: no recursion limit
graph = {"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": []}
stack, seen, order = ["A"], {"A"}, []
while stack:
    node = stack.pop()                   # O(1) from the end
    order.append(node)
    for nxt in graph[node]:
        if nxt not in seen:
            seen.add(nxt)
            stack.append(nxt)
print("DFS order:", order)

01234
recursion limit: 1000
DFS order: ['A', 'C', 'D', 'B']


## 6. Cheat sheet

| I need... | Use | Key cost |
|---|---|---|
| An ordered collection, access by position | `list` | Index O(1); append O(1); front insert/pop O(n) |
| A queue, or BFS | `deque` | Both ends O(1) |
| A stack, or DFS | `list` | `append` / `pop()` O(1) |
| Look up by key | `dict` | O(1) average |
| "Have I seen this?" | `set` | O(1) average |
| Counting | `Counter` | O(n) to build |
| Grouping, adjacency lists | `defaultdict(list)` | O(1) per add |
| The smallest/largest, repeatedly | `heapq` | Push/pop O(log n), peek O(1) |
| Top k of n | heap of size k / `nlargest` | O(n log k) |
| Sort anything | `sorted(..., key=...)` | O(n log n); ~O(n) if nearly sorted |
| Binary search in sorted data | `bisect` | O(log n) search |

And the two questions, one last time:

> **Do I know the address, or must I search?**
> **Does this operation make things move?**

Lists know addresses by position. Dicts and sets turn keys into addresses. Deques never make anything move at their ends. Heaps keep just enough order that only one short path ever moves: see [`heaps.ipynb`](heaps.ipynb).